## Test — Day 4 Silver

In [0]:
# Test — Day 4 Silver
# Runs after silver_dlt_pipeline completes. Checks the actual output,
# including the one thing flagged as unverified in the notebooks: whether
# cars_silver's assumed (location, reading_ts) grain actually holds against
# the real 24.7M-row file.

import unittest

dbutils.widgets.text("catalog_name", "vstone_catalog", "1. Catalog Name")
dbutils.widgets.text("silver_schema", "silver", "2. Silver Schema")
CATALOG = dbutils.widgets.get("catalog_name")
SILVER = dbutils.widgets.get("silver_schema")

SILVER_TABLES = ["streets_silver", "cars_silver", "telegram_silver",
                  "node_locations_silver", "streets_list_silver"]
QUARANTINE_TABLES = [f"{t}_quarantine" for t in SILVER_TABLES]


class SilverDay4Tests(unittest.TestCase):

    def test_all_silver_tables_have_rows(self):
        for t in SILVER_TABLES:
            with self.subTest(table=t):
                count = spark.table(f"{CATALOG}.{SILVER}.{t}").count()
                self.assertGreater(count, 0, f"{t} is empty.")

    def test_all_tables_have_silver_load_dt(self):
        for t in SILVER_TABLES + QUARANTINE_TABLES:
            with self.subTest(table=t):
                df = spark.table(f"{CATALOG}.{SILVER}.{t}")
                missing = df.filter("silver_load_dt IS NULL").count()
                self.assertEqual(missing, 0, f"{t} has {missing} rows missing silver_load_dt.")

    def test_quarantine_tables_have_reason_and_timestamp(self):
        for t in QUARANTINE_TABLES:
            with self.subTest(table=t):
                df = spark.table(f"{CATALOG}.{SILVER}.{t}")
                if df.count() == 0:
                    continue  # empty quarantine is fine (e.g. streets_list had no bad rows)
                missing = df.filter("quarantine_reason IS NULL OR quarantine_dt IS NULL").count()
                self.assertEqual(missing, 0, f"{t} has {missing} rows missing reason/timestamp.")

    def test_streets_silver_grain_is_clean(self):
        """streets: (street_id, reading_ts) confirmed duplicate-free in Day 1 profiling."""
        df = spark.table(f"{CATALOG}.{SILVER}.streets_silver")
        total = df.count()
        distinct = df.select("street_id", "reading_ts").distinct().count()
        self.assertEqual(total, distinct, f"streets_silver grain not clean: {total:,} vs {distinct:,}")

    def test_cars_silver_grain_is_confirmed_clean(self):
        """
        UPDATED 2026-09-17: the original (location, reading_ts) grain
        assumption was WRONG — confirmed via real data to silently drop
        746,345+ distinct readings at location=7 alone (different enter/exit
        values sharing a timestamp, not log duplication). Corrected grain is
        (location, reading_ts, reading_id), confirmed duplicate-free.
        """
        df = spark.table(f"{CATALOG}.{SILVER}.cars_silver")
        total = df.count()
        distinct = df.select("location", "reading_ts", "reading_id").distinct().count()
        self.assertEqual(
            total, distinct,
            f"cars_silver grain (location, reading_ts, reading_id) is NOT clean: "
            f"{total:,} rows vs {distinct:,} distinct triples — the corrected grain "
            f"assumption needs revisiting again."
        )

    def test_node_locations_quarantine_is_exactly_location_7(self):
        """The one known (0,0)-coordinate row should be the only quarantined row."""
        df = spark.table(f"{CATALOG}.{SILVER}.node_locations_silver_quarantine")
        rows = df.select("location").collect()
        locations = {r["location"] for r in rows}
        self.assertEqual(locations, {7}, f"Expected only location=7 quarantined, got: {locations}")

    def test_streets_silver_raining_not_yet_clipped(self):
        """
        Day 4 scope check: raining should still contain values outside
        [0,100] here. If this ever fails, someone applied the Day 5 clip
        rule too early and the day boundary silently blurred.
        """
        df = spark.table(f"{CATALOG}.{SILVER}.streets_silver")
        out_of_range = df.filter("raining < 0 OR raining > 100").count()
        self.assertGreater(out_of_range, 0,
                            "Expected some out-of-range raining values still present at Day 4 — "
                            "clipping is Day 5 scope, not Day 4.")



if __name__ == "__main__":
    suite = unittest.TestLoader().loadTestsFromTestCase(SilverDay4Tests)
    result = unittest.TextTestRunner(verbosity=2).run(suite)
    if not result.wasSuccessful():
        raise Exception("Day 4 silver tests FAILED — see output above.")
